# **AIML Assignment 2 – Problem Solving Through Search**

Name: Yash Sawant

Roll No: 166

PRN: 202401040026

# Common Graph and Heuristic Setup

In [1]:
from collections import deque
import heapq
import random
import math
import time


# -----------------------------
# Common Graph and Heuristics
# -----------------------------

graph = {
    'S': [('A', 2), ('B', 2)],
    'A': [('B', 2), ('C', 2)],
    'B': [('C', 1), ('G', 3)],
    'C': [('G', 2)],
    'G': []
}

heuristic = {
    'S': 5,
    'A': 4,
    'B': 3,
    'C': 2,
    'G': 0
}


# 2.Breadth First Search (BFS)

In [2]:
def bfs(graph, start, goal):
    queue = deque([(start, [start], 0)])
    visited = set()

    while queue:
        node, path, cost = queue.popleft()

        if node == goal:
            return path, cost

        if node in visited:
            continue

        visited.add(node)

        for neighbour, edge_cost in graph[node]:
            if neighbour not in visited:
                queue.append(
                    (neighbour, path + [neighbour], cost + edge_cost)
                )

    return None


# 3. Depth First Search (DFS)

In [3]:
def dfs(graph, start, goal):
    stack = [(start, [start], 0)]
    visited = set()

    while stack:
        node, path, cost = stack.pop()

        if node == goal:
            return path, cost

        if node in visited:
            continue

        visited.add(node)

        for neighbour, edge_cost in reversed(graph[node]):
            if neighbour not in visited:
                stack.append(
                    (neighbour, path + [neighbour], cost + edge_cost)
                )

    return None

# 4. Uniform Cost Search (UCS)

In [4]:
def ucs(graph, start, goal):
    pq = [(0, start, [start])]
    visited = set()

    while pq:
        cost, node, path = heapq.heappop(pq)

        if node == goal:
            return path, cost, len(visited) + 1

        if node in visited:
            continue

        visited.add(node)

        for neighbour, edge_cost in graph[node]:
            if neighbour not in visited:
                heapq.heappush(
                    pq,
                    (cost + edge_cost,
                     neighbour,
                     path + [neighbour])
                )

    return None

# 5. Greedy Best-First Search

In [5]:
def greedy(graph, heuristic, start, goal):
    pq = [(heuristic[start], start, [start], 0)]
    visited = set()

    while pq:
        h, node, path, cost = heapq.heappop(pq)

        if node == goal:
            return path, cost

        if node in visited:
            continue

        visited.add(node)

        for neighbour, edge_cost in graph[node]:
            if neighbour not in visited:
                heapq.heappush(
                    pq,
                    (
                        heuristic[neighbour],
                        neighbour,
                        path + [neighbour],
                        cost + edge_cost
                    )
                )

    return None

# 6. A* Search

In [6]:
def a_star(graph, heuristic, start, goal):
    pq = [(heuristic[start], 0, start, [start])]
    visited = set()

    while pq:
        f, cost, node, path = heapq.heappop(pq)

        if node == goal:
            return path, cost, len(visited) + 1

        if node in visited:
            continue

        visited.add(node)

        for neighbour, edge_cost in graph[node]:
            if neighbour not in visited:
                new_cost = cost + edge_cost
                new_f = new_cost + heuristic[neighbour]

                heapq.heappush(
                    pq,
                    (
                        new_f,
                        new_cost,
                        neighbour,
                        path + [neighbour]
                    )
                )

    return None

# 7. Hill Climbing (8-Queens)

In [7]:
def conflicts(board):
    count = 0
    n = len(board)

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:
                count += 1

            if abs(board[i] - board[j]) == abs(i - j):
                count += 1

    return count

def hill_climbing(n=8):
    board = [random.randint(0, n - 1) for _ in range(n)]
    iterations = 0

    while True:
        current_conflicts = conflicts(board)
        best_board = board[:]
        best_conflicts = current_conflicts

        for row in range(n):
            for col in range(n):
                if board[row] == col:
                    continue

                new_board = board[:]
                new_board[row] = col
                new_conflicts = conflicts(new_board)

                if new_conflicts < best_conflicts:
                    best_board = new_board
                    best_conflicts = new_conflicts

        iterations += 1

        if best_conflicts >= current_conflicts:
            break

        board = best_board

        if best_conflicts == 0:
            break

    return board, conflicts(board), iterations



# 8. Simulated Annealing (8-Queens)

In [8]:
def simulated_annealing(n=8):
    board = [random.randint(0, n - 1) for _ in range(n)]

    current = conflicts(board)
    temperature = 100
    iterations = 0

    while temperature > 0.01 and current > 0:
        new_board = board[:]

        row = random.randint(0, n - 1)
        col = random.randint(0, n - 1)

        new_board[row] = col

        new_conflicts = conflicts(new_board)

        delta = new_conflicts - current

        if delta < 0 or random.random() < math.exp(-delta / temperature):
            board = new_board
            current = new_conflicts

        temperature *= 0.995
        iterations += 1

    return board, conflicts(board), iterations


# 9. Backtracking (4-Queens)

In [9]:
def is_safe(board, row, col):
    for previous_row in range(row):
        previous_col = board[previous_row]

        if previous_col == col:
            return False

        if abs(previous_col - col) == abs(previous_row - row):
            return False

    return True


def solve_4_queens(board, row, count):
    if row == 4:
        return True, count

    for col in range(4):
        count += 1

        if is_safe(board, row, col):
            board[row] = col

            solved, count = solve_4_queens(
                board,
                row + 1,
                count
            )

            if solved:
                return True, count

            board[row] = -1

    return False, count


def backtracking_4_queens():
    board = [-1, -1, -1, -1]

    solved, assignments = solve_4_queens(
        board,
        0,
        0
    )

    return board, assignments, solved

# 10. Run All Algorithms

In [10]:
print("----- SEARCH ALGORITHMS -----")

print("BFS:", bfs(graph, 'S', 'G'))

print("DFS:", dfs(graph, 'S', 'G'))

print("UCS:", ucs(graph, 'S', 'G'))

print("Greedy:", greedy(graph, heuristic, 'S', 'G'))

print("A*:", a_star(graph, heuristic, 'S', 'G'))


print("\n----- LOCAL SEARCH -----")

random.seed(10)

print(
    "Hill Climbing:",
    hill_climbing(8)
)

random.seed(20)

print(
    "Simulated Annealing:",
    simulated_annealing(8)
)


print("\n----- CONSTRAINT SATISFACTION -----")

print(
    "4-Queens CSP:",
    backtracking_4_queens()
)

----- SEARCH ALGORITHMS -----
BFS: (['S', 'B', 'G'], 5)
DFS: (['S', 'A', 'B', 'C', 'G'], 7)
UCS: (['S', 'B', 'C', 'G'], 5, 5)
Greedy: (['S', 'B', 'G'], 5)
A*: (['S', 'B', 'C', 'G'], 5, 4)

----- LOCAL SEARCH -----
Hill Climbing: ([0, 5, 7, 1, 3, 7, 2, 4], 1, 4)
Simulated Annealing: ([3, 1, 7, 5, 0, 2, 4, 6], 0, 1187)

----- CONSTRAINT SATISFACTION -----
4-Queens CSP: ([1, 3, 0, 2], 26, True)
